In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import functools
from math import pi
from multiprocessing.pool import Pool
import itertools

import numpy as np
import scipy
import tqdm.contrib.itertools
import uncertainties
import matplotlib.pyplot as plt
import xarray as xr

from numpy.typing import ArrayLike, NDArray
from fluxoniumcr import DATA_DIR
from fluxoniumcr.qubits.fluxonium import Fluxonium

In [3]:
def calculate_fluxonium_spectrum(
        EJ: float,
        EC: float,
        EL: float,
        EJ_sdev: float,
        EL_sdev: float,
) -> list[uncertainties.Variable]:
    fx = Fluxonium(
        EJ=EJ,
        EC=EC,
        EL=EL,
        dim=16,
        cutoff=128,
    )
    uEJ = uncertainties.ufloat(0.0, std_dev=abs(EJ_sdev))
    uEL = uncertainties.ufloat(0.0, std_dev=abs(EL_sdev))
    cos_phi_op = fx.get_operator("cos_phi")
    phi_op = fx.get_operator("phi")
    phi_squared_op = phi_op @ phi_op
    evals = list(fx.eigenvalues[:4])
    for i in range(len(evals)):
        evals[i] += (
            uEJ * cos_phi_op[i, i]
            + uEL * phi_squared_op[i, i]
        )
    return evals


def calculate_mvn_union_probability(
        mu: ArrayLike,
        cov: ArrayLike,
        lb: ArrayLike,
        ub: ArrayLike,
) -> float:
    # Use inclusion-exclusion principle to calculate the probability of
    # P(Union lb[i] < X[i] < ub[i]) where X[i] are multivariate normally distributed.
    mu = np.asarray(mu)
    cov = np.asarray(cov)
    lb = np.asarray(lb)
    ub = np.asarray(ub)
    
    N = len(mu)
    prob = 0.0
    for k in range(N):
        for idx in map(list, itertools.combinations(range(N), k+1)):
            subset_mu = mu[idx]
            subset_cov = cov[np.ix_(idx, idx)]
            subset_lb = lb[idx]
            subset_ub = ub[idx]
            rv = scipy.stats.multivariate_normal(
                subset_mu,
                subset_cov,
                allow_singular=True,
            )
            prob += (-1)**k * rv.cdf(
                subset_ub,
                lower_limit=subset_lb,
            )
    return prob

In [7]:
# Control qubit energy parameters
EJ_c = 4.0 * 2*pi
EC_c = 1.2 * 2*pi
EL_c = 0.4 * 2*pi

# Target qubit charging and inductive energies
EC_t = 1.0 * 2*pi
EL_t = 1.0 * 2*pi

# Maximum target qubit frequency allowed.
max_qubit_frequency = 1.0 * 2*pi


def fobj(
        x: ArrayLike,
        EJ_rel_sdev: float,
        EL_rel_sdev: float|None = None,
) -> float:    
    if EL_rel_sdev is None:
        EL_rel_sdev = EJ_rel_sdev/10
    
    x = np.asarray(x)
    num_target_qubits = len(x)

    EJ_values = np.array([EJ_c, *sorted(x, reverse=True)])
    EC_values = np.array([EC_c, *(EC_t,)*num_target_qubits])
    EL_values = np.array([EL_c, *(EL_t,)*num_target_qubits])

    spectra: list[list[uncertainties.Variable]] = []

    for EJ, EC, EL in zip(EJ_values, EC_values, EL_values):
        spectrum = calculate_fluxonium_spectrum(
            EJ,
            EC,
            EL,
            EJ_rel_sdev * EJ,
            EL_rel_sdev * EL,
        )
        spectra.append(spectrum)

    f10_all = calculate_qubit_frequencies(x, EJ_rel_sdev, EL_rel_sdev)
    f10_c = f10_all[0]
    f10_ts = f10_all[1:]

    deltas: list[list[uncertainties.Variable]] = []

    # Control and target qubit frequency collision.
    deltas.append([
        f10_c - f10_ts[0]
    ])
    
    # Target qubit frequencies collision.
    deltas.append([
        f10_ts[i] - f10_ts[i-1]
        for i in range(1, num_target_qubits)
    ])
    
    # Control and spectator 00 -> 11 collision.
    deltas.append([
        f10_c + f10_ts[j] - 2*f10_ts[i]
        for i in range(2)
        for j in range(i+1, num_target_qubits)
    ])
    
    # Keep target qubit frequency below max_qubit_frequency.
    deltas.append([
        max_qubit_frequency - f10_ts[-1]
    ])

    # Predetermined collision windows.
    windows = 2*pi * np.array([
        (-80e-3, np.inf),
        (-35e-3, +35e-3),
        (-40e-3, +20e-3),
        (-np.inf, 0.0),
    ])

    deltas_flat = np.array(
        list(itertools.chain.from_iterable(deltas))
    )
    windows_flat = np.array(
        list(itertools.chain.from_iterable(
            len(d) * [w] for d, w in zip(deltas, windows)
        ))
    )

    mu = np.array([d.nominal_value for d in deltas_flat])
    cov = np.array(uncertainties.covariance_matrix(deltas_flat))
    prob = calculate_mvn_union_probability(
        mu,
        cov,
        lb=windows_flat[:, 0],
        ub=windows_flat[:, 1],
    )
    
    return prob


def calculate_qubit_frequencies(
        x: ArrayLike,
        EJ_rel_sdev: float,
        EL_rel_sdev: float|None = None,
) -> list[uncertainties.Variable]:
    if EL_rel_sdev is None:
        EL_rel_sdev = EJ_rel_sdev/10
    
    x = np.asarray(x)
    num_target_qubits = len(x)

    EJ_values = np.array([EJ_c, *sorted(x, reverse=True)])
    EC_values = np.array([EC_c, *(EC_t,)*num_target_qubits])
    EL_values = np.array([EL_c, *(EL_t,)*num_target_qubits])

    spectra: list[list[uncertainties.Variable]] = []

    for EJ, EC, EL in zip(EJ_values, EC_values, EL_values):
        spectrum = calculate_fluxonium_spectrum(
            EJ,
            EC,
            EL,
            EJ_rel_sdev * EJ,
            EL_rel_sdev * EL,
        )
        spectra.append(spectrum)

    f10_all = [s[1] - s[0] for s in spectra]
    
    return f10_all

In [49]:
num_target_qubits = 4
EJ_min = 3.0 * 2*pi
EJ_max = 6.0 * 2*pi
EJ_rel_sdev_data = np.linspace(0.01, 0.10, 91)

EJ_rel_sdev_coord = xr.DataArray(
    EJ_rel_sdev_data,
    dims=['EJ_rel_sdev']
)
qubit_coord = xr.DataArray(
    np.arange(num_target_qubits, dtype=np.int8),
    dims=['qubit'],
)
dataset = xr.Dataset(
    attrs=dict(
        EJ_control=EJ_c,
        EC_control=EC_c,
        EL_control=EL_c,
        EC_target=EC_t,
        EL_target=EL_t,
        EJ_min=EJ_min,
        EJ_max=EJ_max,
        max_qubit_frequency=max_qubit_frequency,
    )
)
dataset['EJ'] = xr.DataArray(
    np.nan,
    coords=[EJ_rel_sdev_coord, qubit_coord],
    attrs=dict(
        unit="Grad/s"
    )   
)

dataset['collision_probability'] = xr.DataArray(
    np.nan,
    coords=[EJ_rel_sdev_coord],
)

dataset['f10'] = xr.DataArray(
    np.nan,
    coords=[dataset.EJ_rel_sdev, dataset.qubit],
    attrs=dict(
        unit="Grad/s"
    )   
)

dataset['f10_sdev'] = xr.DataArray(
    np.nan,
    coords=[dataset.EJ_rel_sdev, dataset.qubit],
    attrs=dict(
        unit="Grad/s"
    )   
)
dataset['f10c'] = xr.DataArray(
    np.nan,
    coords=[dataset.EJ_rel_sdev],
    attrs=dict(
        unit="Grad/s"
    )   
)

dataset['f10c_sdev'] = xr.DataArray(
    np.nan,
    coords=[dataset.EJ_rel_sdev],
    attrs=dict(
        unit="Grad/s"
    )   
)

In [64]:
def doit(
        EJ_rel_sdev: float,
        B: scipy.optimize.Bounds,
        C: scipy.optimize.LinearConstraint,
        iters: int = 64,
) -> tuple[float, NDArray[np.floating]]:
    res = scipy.optimize.shgo(
        fobj,
        bounds=B,
        constraints=C,
        args=(EJ_rel_sdev,),
        minimizer_kwargs=dict(
            method='COBYQA',
        ),
        iters=iters,
        sampling_method='simplicial',
    )
    f10_all = calculate_qubit_frequencies(res.x, EJ_rel_sdev)
    return res.x, res.fun, f10_all


B = scipy.optimize.Bounds(
    [EJ_min] * num_target_qubits,
    [EJ_max] * num_target_qubits,
)

# Constraint to enforce EJ are in descending order.
C = scipy.optimize.LinearConstraint(
    np.eye(num_target_qubits - 1, num_target_qubits, k=0)\
        - np.eye(num_target_qubits - 1, num_target_qubits, k=1),
    lb=0,
)

with Pool(processes=8) as pool:
    mask = slice(None, None)
    mask = [77]  # Recalculate these indices
    for EJ_rel_sdev, return_values in tqdm.contrib.tzip(
            dataset.EJ_rel_sdev.data[mask],
            pool.imap(
                functools.partial(
                    doit,
                    B=B,
                    C=C,
                    iters=256,  # Higher iterations to find global minimum
                ),
                dataset.EJ_rel_sdev.data[mask],
            )
    ):
        EJ, collision_probability, f10_all = return_values
        ds = dataset.loc[dict(EJ_rel_sdev=EJ_rel_sdev)]
        ds['EJ'][:] = EJ
        ds['collision_probability'][()] = collision_probability
        ds['f10'][:] = [v.nominal_value for v in f10_all[1:]]
        ds['f10_sdev'][:] = [v.std_dev for v in f10_all[1:]]
        ds['f10c'][()] = f10_all[0].nominal_value
        ds['f10c_sdev'][()] = f10_all[0].std_dev

  0%|          | 0/1 [00:00<?, ?it/s]

In [67]:
parent_path = (
    DATA_DIR
    /"EJ_allocation"
    / (
        f"EJ_control={EJ_c/(2*pi):.2f},"
        f"EC_control={EC_c/(2*pi):.2f},"
        f"EL_control={EL_c/(2*pi):.2f}"
    )
)
parent_path.mkdir(parents=True, exist_ok=True)

dataset_filename = (
    f"EC_target={EC_t/(2*pi):.2f},"
    f"EL_target={EL_t/(2*pi):.2f},"
    f"num_targets={num_target_qubits}"
    ".hdf5"
)
print(f"Saving to '{dataset_filename}'")
dataset.to_netcdf(parent_path/dataset_filename)

Saving to 'EC_target=1.00,EL_target=1.00,num_targets=4.hdf5'
